# PROYECTO DE ANALÍTICA DE DATOS - MACI 2026
**Alumno:** Keyden Urbina P.

**Dataset:** Amazon Products Sales Dataset 2025

**Fuente:** https://www.kaggle.com/datasets/ikramshah512/amazon-products-sales-dataset-42k-items-2025?resource=download

## Búsqueda de ASIN (identificador único de producto)

**Para qué sirve este notebook (explicado simple):**
En Amazon cada producto tiene un código único de 10 caracteres llamado **ASIN** (ej. `B0D54JZTHY`). El dataset no lo trae como columna, pero está escondido dentro de la URL del producto (`.../dp/B0D54JZTHY/...`). Con el ASIN podemos saber cuándo varias filas son en realidad el **mismo producto** (el dataset tiene 42.675 filas pero muchos menos productos).

**Qué hace, en 4 pasos:**
1. Lee el CSV limpio y extrae el ASIN desde la URL de cada fila.
2. Lista los productos que quedaron sin ASIN (no tenían URL) y genera un link de búsqueda en Amazon para cada uno.
3. Aplica los ASIN que se buscaron y confirmaron **a mano** en Amazon (tabla `VALIDADOS`), con un control: solo se aplican si el número de filas coincide con lo esperado.
4. Guarda el dataset con ASIN y una tabla de trazabilidad (qué ASIN se agregó, de dónde salió y cómo se validó).

Cómo usarlo: *Run → Run All*. Para incorporar un nuevo ASIN validado, basta con agregar una fila a `VALIDADOS` (Paso 3) y volver a correr todo.

### Paso 1 -- Cargar datos y extraer el ASIN desde la URL

**Qué hace:**
- `extraer_asin` primero "decodifica" la URL (las URL de anuncios patrocinados traen `/` escrito como `%2F`) y luego busca el patrón `/dp/XXXXXXXXXX` o `/gp/product/XXXXXXXXXX` (10 letras o números).
- Si la URL está vacía o no tiene ese patrón, el ASIN queda vacío (`<NA>`).
- `pendientes`: productos sin ASIN, agrupados por título + imagen, ordenados de más a menos filas. `buscar_amazon` es un link listo para buscar el producto en Amazon.

In [2]:
import re
import pandas as pd
from pathlib import Path
from urllib.parse import unquote, quote_plus
from IPython.display import display

pd.set_option("display.max_colwidth", None)

# Nota: si tu notebook está dentro de 'notebooks/', '../' sube a la raíz del proyecto.
CARPETA = Path("../data/procesado")
ruta_csv = CARPETA / "amazon_electronics_data_cleaned.csv"
ruta_salida = CARPETA / "amazon_electronics_cleaned_with_asin.csv"
ruta_validaciones = CARPETA / "asin_pendientes_validacion.csv"

df = pd.read_csv(ruta_csv)
print(f"Dataset cargado con éxito: {df.shape[0]:,} filas | {df.shape[1]} columnas")

def extraer_asin(url):
    if pd.isna(url):
        return pd.NA
    resultado = re.search(r"/(?:dp|gp/product)/([A-Z0-9]{10})(?=[/?#]|$)", unquote(str(url)), flags=re.I)
    return resultado.group(1).upper() if resultado else pd.NA

df["asin"] = df["product_page_url"].apply(extraer_asin)

def productos_pendientes(claves):
    "Productos sin ASIN agrupados por 'claves', con cantidad de filas y link de búsqueda."
    p = (df[df["asin"].isna()].groupby(claves, dropna=False).size()
         .reset_index(name="registros").sort_values("registros", ascending=False))
    p["buscar_amazon"] = p["product_title"].apply(
        lambda x: "https://www.amazon.com/s?k=" + quote_plus(str(x)) if pd.notna(x) else "")
    return p

pendientes = productos_pendientes(["product_title", "product_image_url"])
print("ASIN identificados:", df["asin"].notna().sum())
print("Productos pendientes:", len(pendientes))
display(pendientes[["product_title", "registros", "buscar_amazon"]].head(10))

Dataset cargado con éxito: 42,675 filas | 17 columnas
ASIN identificados: 40606
Productos pendientes: 189


,product_title,registros,buscar_amazon
62,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,https://www.amazon.com/s?k=Duracell+Coppertop+9V+Battery%2C+6+Count+%28Pack+of+1%29%2C+9+Volt+Battery+with+Long-lasting+Power%2C+All-Purpose+Alkaline+9V+Battery+for+Household+and+Office+Devices
129,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,https://www.amazon.com/s?k=Energizer+MAX+AA+Batteries+%2816+Pack%29%2C+Double+A+Alkaline+Batteries
32,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,https://www.amazon.com/s?k=CR+2016+MAXELL+LITHIUM+BATTERIES+%282+piece%29+3V+Watch+2016+New
111,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,https://www.amazon.com/s?k=Energizer+Alkaline+Power+AAA+Batteries%2C+32+Count%2C+Long-Lasting+Triple+A+Batteries%2C+Suitable+for+Everyday+Electronics+and+Emergency+Gear
76,Duracell Precharged Recharg. Battery AA (Pack of 2),79,https://www.amazon.com/s?k=Duracell+Precharged+Recharg.+Battery+AA+%28Pack+of+2%29
186,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,https://www.amazon.com/s?k=Renata+Watch+Battery+321+0%25Hg+Mercury+Free+X+3
188,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65,https://www.amazon.com/s?k=Windex+Electronic+Cleaning+Wipes%2C+Pre-Moistened+Screen+Wipes+for+TVs%2C+Phones%2C+and+Other+Electronics%2C+Resealable+for+a+Freshness+Lock%2C+25+Count+%28Pack+of+3%29
120,"Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",19,https://www.amazon.com/s?k=Energizer+CR2032+Lithium+3V+Coin+Batteries+for+Watches%2C+AirTags%2C+Key+Fobs%2C+Small+Electronics+%2810+Pack%29
133,"Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries",14,https://www.amazon.com/s?k=Energizer+MAX+AAA+Batteries+%2816+Pack%29%2C+Triple+A+Alkaline+Batteries
104,"Energizer AA and AAA Batteries, 48 Count, Combo Pack Contains 24 Max Double A and 24 Max Triple A Batteries",6,https://www.amazon.com/s?k=Energizer+AA+and+AAA+Batteries%2C+48+Count%2C+Combo+Pack+Contains+24+Max+Double+A+and+24+Max+Triple+A+Batteries


### Paso 2 -- ¿Se puede recuperar algún ASIN sin buscar en Amazon?

**Qué hace:** compara cada producto pendiente con los productos que **sí** tienen ASIN:
- **Cruce 1:** mismo título **y** misma imagen.
- **Cruce 2:** (para los que no calzaron) mismo título, aunque cambie la imagen.

Solo se considera recuperable si el cruce apunta a **un único** ASIN (si apunta a dos o más, es ambiguo).
Este paso es solo un **diagnóstico**: muestra candidatos, pero no los aplica (igual que en la versión original).

In [3]:
def candidatos(pend, claves):
    catalogo = (df[df["asin"].notna()].groupby(claves, dropna=False)["asin"]
                .agg(asin_distintos="nunique", asin_posible="first").reset_index())
    return pend.merge(catalogo, on=claves, how="left")

cruce1 = candidatos(pendientes, ["product_title", "product_image_url"])
cruce2 = candidatos(cruce1.loc[cruce1["asin_distintos"] != 1, ["product_title", "product_image_url", "registros"]],
                    "product_title")

for nombre, cruce in [("Cruce 1 - título + imagen", cruce1), ("Cruce 2 - solo título", cruce2)]:
    ok = cruce[cruce["asin_distintos"] == 1]
    print(f"\n{nombre}: {len(ok)} grupos | {ok['registros'].sum()} registros potencialmente recuperables")
    display(ok[["product_title", "registros", "asin_posible"]].head(20))


Cruce 1 - título + imagen: 1 grupos | 1 registros potencialmente recuperables


,product_title,registros,asin_posible
40,Renata 371 SR920SW Batteries - 1.55V Silver Oxide 371 Watch Battery (2 Count),1,B00MU3FWUS



Cruce 2 - solo título: 1 grupos | 1 registros potencialmente recuperables


,product_title,registros,asin_posible
134,"ACDelco 48-Count Triple AAA Batteries, Maximum Power Super Alkaline Battery for Home and Office, 10-Year Shelf Life, Reclosable Packaging",1,B00FF8KQJC


### Paso 3 -- ASIN validados manualmente en Amazon

**Qué hace:** esta tabla es el registro de los ASIN que se buscaron en Amazon y se confirmaron a mano. Cada fila dice: título exacto del producto, cuántas filas del dataset se espera actualizar, el ASIN confirmado, el método y el estado.

El link `fuente` se arma solo a partir del ASIN.

In [4]:
VALIDADOS = [
    # (título exacto, filas esperadas, ASIN, método, estado)
    ("CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New",
     204, "B00IX9OUTM", "Búsqueda externa", "Validado"),
    ("Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",
     744, "B0B72DBKVF", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",
     566, "B004RFF57C", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",
     93, "B09RTVN5GN", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Renata Watch Battery 321 0%Hg Mercury Free X 3",
     79, "B00NHYB6VI", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Duracell Precharged Recharg. Battery AA (Pack of 2)",
     79, "B081J2QH83", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",
     65, "B072B91QTN", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",
     19, "B0CH1N7X5W", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries",
     14, "B0D51CSMKX", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer AA and AAA Batteries, 48 Count, Combo Pack Contains 24 Max Double A and 24 Max Triple A Batteries",
     6, "B087GMJXMC", "Verificación externa Amazon", "Validado (ficha actual)"),
    ("Energizer AA Batteries Alkaline Power, 32 Count Alkaline Double A Battery Pack",
     6, "B09RTVD1GF", "Verificación externa Amazon", "Validado (ficha actual)"),
]

asin_pendientes_validacion = pd.DataFrame(VALIDADOS, columns=["product_title", "registros", "asin_candidato", "metodo", "estado"])
asin_pendientes_validacion.insert(3, "fuente", "https://www.amazon.com/dp/" + asin_pendientes_validacion["asin_candidato"])

# Control de calidad: cada ASIN debe tener exactamente 10 letras/números
assert asin_pendientes_validacion["asin_candidato"].str.fullmatch(r"[A-Z0-9]{10}").all(), "Hay un ASIN mal escrito"
display(asin_pendientes_validacion)

,product_title,registros,asin_candidato,fuente,metodo,estado
0,CR 2016 MAXELL LITHIUM BATTERIES (2 piece) 3V Watch 2016 New,204,B00IX9OUTM,https://www.amazon.com/dp/B00IX9OUTM,Búsqueda externa,Validado
1,"Duracell Coppertop 9V Battery, 6 Count (Pack of 1), 9 Volt Battery with Long-lasting Power, All-Purpose Alkaline 9V Battery for Household and Office Devices",744,B0B72DBKVF,https://www.amazon.com/dp/B0B72DBKVF,Verificación externa Amazon,Validado (ficha actual)
2,"Energizer MAX AA Batteries (16 Pack), Double A Alkaline Batteries",566,B004RFF57C,https://www.amazon.com/dp/B004RFF57C,Verificación externa Amazon,Validado (ficha actual)
3,"Energizer Alkaline Power AAA Batteries, 32 Count, Long-Lasting Triple A Batteries, Suitable for Everyday Electronics and Emergency Gear",93,B09RTVN5GN,https://www.amazon.com/dp/B09RTVN5GN,Verificación externa Amazon,Validado (ficha actual)
4,Renata Watch Battery 321 0%Hg Mercury Free X 3,79,B00NHYB6VI,https://www.amazon.com/dp/B00NHYB6VI,Verificación externa Amazon,Validado (ficha actual)
5,Duracell Precharged Recharg. Battery AA (Pack of 2),79,B081J2QH83,https://www.amazon.com/dp/B081J2QH83,Verificación externa Amazon,Validado (ficha actual)
6,"Windex Electronic Cleaning Wipes, Pre-Moistened Screen Wipes for TVs, Phones, and Other Electronics, Resealable for a Freshness Lock, 25 Count (Pack of 3)",65,B072B91QTN,https://www.amazon.com/dp/B072B91QTN,Verificación externa Amazon,Validado (ficha actual)
7,"Energizer CR2032 Lithium 3V Coin Batteries for Watches, AirTags, Key Fobs, Small Electronics (10 Pack)",19,B0CH1N7X5W,https://www.amazon.com/dp/B0CH1N7X5W,Verificación externa Amazon,Validado (ficha actual)
8,"Energizer MAX AAA Batteries (16 Pack), Triple A Alkaline Batteries",14,B0D51CSMKX,https://www.amazon.com/dp/B0D51CSMKX,Verificación externa Amazon,Validado (ficha actual)
9,"Energizer AA and AAA Batteries, 48 Count, Combo Pack Contains 24 Max Double A and 24 Max Triple A Batteries",6,B087GMJXMC,https://www.amazon.com/dp/B087GMJXMC,Verificación externa Amazon,Validado (ficha actual)


### Paso 4 -- Incorporar los ASIN validados al dataset

**Qué hace:**
1. Crea la columna `asin_origen` para saber de dónde salió cada ASIN: `"URL original"` (extraído en el Paso 1) o `"Validación manual Amazon"` (de la tabla del Paso 3).
2. Para cada producto validado busca las filas con ese título exacto **que aún no tienen ASIN**.
3. **Control:** si la cantidad de filas encontradas no es la esperada, se detiene con un error y no se modifica nada (protege contra títulos mal copiados o datos que cambiaron).

In [5]:
df["asin_origen"] = pd.NA
df.loc[df["asin"].notna(), "asin_origen"] = "URL original"

filtros = {}
for v in asin_pendientes_validacion.itertuples():
    filtro = df["product_title"].eq(v.product_title) & df["asin"].isna()
    if filtro.sum() != v.registros:
        raise ValueError(f"{v.asin_candidato}: se esperaban {v.registros} registros y hay {filtro.sum()}. No se modificó nada.")
    filtros[v.asin_candidato] = filtro

for asin, filtro in filtros.items():          # solo se escribe si TODOS los controles pasaron
    df.loc[filtro, "asin"] = asin
    df.loc[filtro, "asin_origen"] = "Validación manual Amazon"
    print(f"{asin}: {filtro.sum()} registros actualizados")

B00IX9OUTM: 204 registros actualizados
B0B72DBKVF: 744 registros actualizados
B004RFF57C: 566 registros actualizados
B09RTVN5GN: 93 registros actualizados
B00NHYB6VI: 79 registros actualizados
B081J2QH83: 79 registros actualizados
B072B91QTN: 65 registros actualizados
B0CH1N7X5W: 19 registros actualizados
B0D51CSMKX: 14 registros actualizados
B087GMJXMC: 6 registros actualizados
B09RTVD1GF: 6 registros actualizados


### Paso 5 -- Guardar y resumen final

**Qué hace:** guarda los dos archivos (dataset con ASIN y tabla de validaciones) y muestra la cobertura: % de filas que ya tienen ASIN. `utf-8-sig` es una codificación que Excel abre bien con tildes y ñ.
Al final lista los títulos que siguen pendientes, para continuar la búsqueda manual.

In [6]:
df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
asin_pendientes_validacion.to_csv(ruta_validaciones, index=False, encoding="utf-8-sig")

print("Archivos guardados:", ruta_salida, "|", ruta_validaciones)
print("Total de registros:", len(df))
print("Registros con ASIN:", df["asin"].notna().sum())
print("Registros pendientes:", df["asin"].isna().sum())
print("ASIN únicos:", df["asin"].nunique())
print("Cobertura:", round(df["asin"].notna().mean() * 100, 2), "%")

pendientes = productos_pendientes("product_title")
print("\nTítulos pendientes:", len(pendientes))
display(pendientes.head(15))

Archivos guardados: ../data/procesado/amazon_electronics_cleaned_with_asin.csv | ../data/procesado/asin_pendientes_validacion.csv
Total de registros: 42675
Registros con ASIN: 42481
Registros pendientes: 194
ASIN únicos: 8641
Cobertura: 99.55 %

Títulos pendientes: 178


,product_title,registros,buscar_amazon
104,"Energizer Alkaline Power AA Batteries (20 Pack), Double A Batteries Alkaline, Long-Lasting",4,https://www.amazon.com/s?k=Energizer+Alkaline+Power+AA+Batteries+%2820+Pack%29%2C+Double+A+Batteries+Alkaline%2C+Long-Lasting
35,Duracell - 2450 3V Lithium Coin Battery - long lasting battery - 6 Count (Pack of 1),3,https://www.amazon.com/s?k=Duracell+-+2450+3V+Lithium+Coin+Battery+-+long+lasting+battery+-+6+Count+%28Pack+of+1%29
89,"Energizer 30039800039768 Max C Batteries, Premium Alkaline C Cell Batteries (4 Battery Count), Silver",2,https://www.amazon.com/s?k=Energizer+30039800039768+Max+C+Batteries%2C+Premium+Alkaline+C+Cell+Batteries+%284+Battery+Count%29%2C+Silver
99,Energizer AA Rechargeable batteries NiMH 2300 mAh 1.2V NH15 - 12 Count,2,https://www.amazon.com/s?k=Energizer+AA+Rechargeable+batteries+NiMH+2300+mAh+1.2V+NH15+-+12+Count
112,"Energizer CR2025 Batteries, 3V Lithium Coin Cell 2025 Watch Battery, (6 Count)",2,https://www.amazon.com/s?k=Energizer+CR2025+Batteries%2C+3V+Lithium+Coin+Cell+2025+Watch+Battery%2C+%286+Count%29
118,"Energizer EN93 Industrial C 12 Alkaline Batteries, Pack of 12",2,https://www.amazon.com/s?k=Energizer+EN93+Industrial+C+12+Alkaline+Batteries%2C+Pack+of+12
160,"Powermax 12-Count 9V Batteries, Ultra Long Lasting Alkaline Battery for Home and Office, 7-Year Shelf Life, Reclosable Packaging",2,https://www.amazon.com/s?k=Powermax+12-Count+9V+Batteries%2C+Ultra+Long+Lasting+Alkaline+Battery+for+Home+and+Office%2C+7-Year+Shelf+Life%2C+Reclosable+Packaging
115,"Energizer D Batteries, D Cell Battery Premium Alkaline, 8 Count",2,https://www.amazon.com/s?k=Energizer+D+Batteries%2C+D+Cell+Battery+Premium+Alkaline%2C+8+Count
116,"Energizer EN55-12IT, 12 Volt 55 Amp Hr, Long Lasting Sealed Lead Acid AGM Rechargeable Battery Ideal for Use in Electric Scooters, Wheelchairs, Emergency Lighting, Backup Power, and More",2,https://www.amazon.com/s?k=Energizer+EN55-12IT%2C+12+Volt+55+Amp+Hr%2C+Long+Lasting+Sealed+Lead+Acid+AGM+Rechargeable+Battery+Ideal+for+Use+in+Electric+Scooters%2C+Wheelchairs%2C+Emergency+Lighting%2C+Backup+Power%2C+and+More
96,Energizer 80 AAA Ultimate Lithium Long Lasting Leakproof Batteries,2,https://www.amazon.com/s?k=Energizer+80+AAA+Ultimate+Lithium+Long+Lasting+Leakproof+Batteries
